# Problem 3: Protein Docking - HA and HD100 Complex

## Problem Summary
Perform structural modeling of the HA-HD100 complex using both unbiased and biased protein docking approaches.

---

## Biological Background

### What is Hemagglutinin (HA)?
**Hemagglutinin (HA)** is the major surface glycoprotein of influenza viruses. It plays two critical roles:
1. **Receptor binding**: HA binds to sialic acid receptors on host cell surfaces
2. **Membrane fusion**: After endocytosis, HA mediates fusion of viral and endosomal membranes

HA is the primary target for neutralizing antibodies and the main component of influenza vaccines. It's a **homotrimer** where each monomer has two subunits (HA1 and HA2) connected by a disulfide bond.

### What is HD100?
**HD100** is a **computationally designed protein** based on the scaffold of **1U84** (a de novo designed three-helix bundle). Key points:
- HD100 was designed to bind the **conserved stem region** of HA
- It was created by introducing specific mutations to 1U84 that create a complementary binding interface for HA
- It targets the **same epitope** as the broadly neutralizing antibody **CR6261**
- HD100 acts as an **antibody mimetic** - it can neutralize influenza by blocking HA conformational changes

### Why Docking Matters
Protein-protein docking predicts how two proteins interact structurally. For HA-HD100:
- **Unbiased docking**: Explores all possible binding orientations - useful when binding site is unknown
- **Biased docking**: Uses known interface residues as restraints - produces more accurate predictions when epitope information is available

### Competition with Fab CR6261
The provided structure **HA_Ab.pdb** contains HA in complex with Fab CR6261 antibody. Since HD100 was designed to:
- Bind the **same HA epitope** as CR6261
- **Compete** with CR6261 for HA binding

We can use the HA-Fab interface residues as restraints for HD100 docking!

---

## Points Distribution

### Part A - Unbiased Docking (2.0 pts)
- A1. Obtain HA and HD100 coordinates: 0.5 + 1.5 pts
- A2. Perform unbiased docking: 1.0 pts
- A3. Extract top 10 poses: 0.5 pts

### Part B - Biased Docking (5.0 pts)
- B1. Identify interface residues: 2.0 pts
- B2. Prepare restraint files: 1.0 pts
- B3. Prepare params and run docking: 0.5 + 1.0 pts
- B4. Extract top 10 poses: 0.5 pts

**Total: 8.5 points**

---
## Configuration

In [1]:
# ============================================================
# CONFIGURATION - Modify these paths as needed
# ============================================================

# Input files (provided in Exam/files/)
HA_AB_FILE = "Exam/files/HA_Ab.pdb"      # HA-Antibody complex
HD100_SEQ_FILE = "Exam/files/HD100.fa"   # HD100 sequence
TEMPLATE_1U84 = "Exam/files/1U84.pdb"    # Template for HD100 modeling

# Output directories
OUTPUT_DIR = "Problem_3_outputs"
SCRATCH_DIR = "Problem_3_outputs/scratch"
RESULTS_DIR = "Problem_3_outputs/results"

# Database paths
SWISSPROT_DB = "databases/swissprot/swissprot"
PDBAA_DB = "databases/pdb_seq/pdbaa"
PFAM_DB = "databases/hmm/Pfam/Pfam-A.hmm"

# Working directories
TEMP_DIR = "temp"
TEMPLATES_DIR = "Templates"
MODELLER_DIR = "Modeller_Templates"

In [2]:
import os
import sys
import subprocess
from pathlib import Path
import pandas as pd
import numpy as np

# Add src to path
sys.path.insert(0, str(Path.cwd()))

# Import project modules
from src.Homology.superimpose import superimpose_structures, write_structure
from src.modeller.scripts import (
    generate_single_template_script,
    ModellerRunner
)
from src.Analysis.visualization import compare_structures

from Bio import SeqIO, pairwise2
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
from Bio.PDB import PDBParser, PDBIO, Select, PPBuilder, Superimposer

# Create output directories
for d in [OUTPUT_DIR, SCRATCH_DIR, RESULTS_DIR]:
    os.makedirs(d, exist_ok=True)

print(f"Output directory: {OUTPUT_DIR}")
print(f"Scratch directory: {SCRATCH_DIR}")
print(f"Results directory: {RESULTS_DIR}")

/opt/miniconda3/envs/Modeller/lib/python3.11/site-packages/Bio/pairwise2.py:278: BiopythonDeprecationWarning: Bio.pairwise2 has been deprecated, and we intend to remove it in a future release of Biopython. As an alternative, please consider using Bio.Align.PairwiseAligner as a replacement, and contact the Biopython developers if you still need the Bio.pairwise2 module.
  warnings.warn(


Output directory: Problem_3_outputs
Scratch directory: Problem_3_outputs/scratch
Results directory: Problem_3_outputs/results


In [3]:
# Verify input files exist
input_files = [
    (HA_AB_FILE, "HA-Antibody complex"),
    (HD100_SEQ_FILE, "HD100 sequence"),
    (TEMPLATE_1U84, "1U84 template")
]

print("=== Input Files ===")
for filepath, description in input_files:
    status = "[x]" if Path(filepath).exists() else "[ ] MISSING"
    print(f"{status} {filepath} - {description}")

=== Input Files ===
[x] Exam/files/HA_Ab.pdb - HA-Antibody complex
[x] Exam/files/HD100.fa - HD100 sequence
[x] Exam/files/1U84.pdb - 1U84 template


---
# Part A: Unbiased Docking

## A1a. Extract HA Receptor Coordinates (0.5 pts)

### What You Need to Do
Extract the HA receptor chain from the HA_Ab.pdb complex file.

### How to Identify Which Chain is HA
1. **By size**: HA is typically the largest chain (~500 residues), while Fab chains are smaller (~150 residues each)
2. **By sequence**: HA starts with characteristic residues (varies by strain)
3. **By chain naming convention**: Often HA is chain A, Fab heavy chain is H, light chain is L

### Biological Context
- HA_Ab.pdb contains HA bound to Fab CR6261 (a broadly neutralizing antibody)
- We need to extract **ONLY** the HA chain for docking
- The Fab chains (H and L) should NOT be included - HD100 will be docked in place of the antibody

In [4]:
# Parse the HA-Antibody complex
parser = PDBParser(QUIET=True)
ha_ab_structure = parser.get_structure('HA_Ab', HA_AB_FILE)

# List all chains to identify HA vs Fab
print("=== Chains in HA_Ab.pdb ===")
for model in ha_ab_structure:
    for chain in model:
        residues = [r for r in chain.get_residues() if r.id[0] == ' ']
        print(f"Chain {chain.id}: {len(residues)} residues")
        
        # Get sequence
        ppb = PPBuilder()
        for pp in ppb.build_peptides(chain):
            seq = str(pp.get_sequence())
            print(f"  First 50 aa: {seq[:50]}...")
            break

=== Chains in HA_Ab.pdb ===
Chain A: 503 residues
  First 50 aa: ADPGDTICIGYHANNSTDTVDTVLEKN...
Chain H: 160 residues
  First 50 aa: EVQLVESGAEVKKPGSSVKVSCKASGGPFRSYAISWVRQAPGQGPEWMGG...
Chain L: 151 residues
  First 50 aa: VLTQPPS...


In [5]:
# Chain selection class
class ChainSelect(Select):
    def __init__(self, chain_ids):
        self.chain_ids = chain_ids if isinstance(chain_ids, list) else [chain_ids]
    
    def accept_chain(self, chain):
        return chain.id in self.chain_ids

# NOTE: Usually HA is the larger chain (receptor)
# Fab consists of Heavy (H) and Light (L) chains
# Adjust HA_CHAIN based on the output above

HA_CHAIN = 'A'  # Modify based on actual chain identification

# Extract HA chain
ha_output = Path(RESULTS_DIR) / "HA.pdb"
io = PDBIO()
io.set_structure(ha_ab_structure)
io.save(str(ha_output), ChainSelect(HA_CHAIN))

print(f"Saved HA receptor to: {ha_output}")

Saved HA receptor to: Problem_3_outputs/results/HA.pdb


## A1b. Model HD100 Structure (1.5 pts)

### What You Need to Do
Build a homology model of HD100 using 1U84 as the template.

### Why Use 1U84 as Template?
HD100 was **designed based on 1U84**:
- 1U84 is a de novo designed three-helix bundle (HB36.3)
- HD100 contains mutations that create HA-binding interface
- The overall fold remains the same, making 1U84 an ideal template
- Sequence identity should be very high (>90%)

### Identifying the Mutations
The mutations introduced in HD100 compared to 1U84 are **critical** because:
1. These mutations create the **binding interface** with HA
2. They will be used as **restraints** for biased docking (Part B)
3. Understanding which positions changed tells us where HD100 contacts HA

### How to Build the Model
1. Align HD100 sequence to 1U84 template sequence
2. Create a PIR alignment file for MODELLER
3. Run MODELLER to generate 5 models
4. Select the best model based on DOPE score (lowest is best)

In [6]:
# Read HD100 sequence
hd100_record = SeqIO.read(HD100_SEQ_FILE, 'fasta')
hd100_seq = str(hd100_record.seq)

print(f"HD100 sequence ({len(hd100_seq)} aa):")
print(hd100_seq)

HD100 sequence (85 aa):
GQQLNRLLLEWIGAWDPFGLGKDAYDVEAEAVLQAVYETESAFDLAMRIMWIYVFAFNRPIPFSHAQKLARRLLELKQAASSPLP


In [7]:
# Read 1U84 template sequence
template_structure = parser.get_structure('1U84', TEMPLATE_1U84)

# Extract sequence from template
ppb = PPBuilder()
template_seqs = []
for model in template_structure:
    for chain in model:
        for pp in ppb.build_peptides(chain):
            template_seqs.append((chain.id, str(pp.get_sequence())))

print("=== 1U84 Template Sequences ===")
for chain_id, seq in template_seqs:
    print(f"Chain {chain_id} ({len(seq)} aa): {seq}")

# Use the first/main chain
template_chain = template_seqs[0][0]
template_seq = template_seqs[0][1]

=== 1U84 Template Sequences ===
Chain A (81 aa): GQQLNRLLLEWIGAWDPFGLGKDAYDVEAASVLQAVYETEDARTLAARIQSIYEFAFDEPIPFPHCLKLARRLLELKQAAS


In [ ]:
# Align HD100 with 1U84 to identify mutations
alignments = pairwise2.align.globalms(hd100_seq, template_seq, 2, -1, -0.5, -0.1)
best_alignment = alignments[0]

aligned_hd100, aligned_1u84, score, begin, end = best_alignment

print("=== Sequence Alignment ===")
print(f"Score: {score}")
print(f"\nHD100:  {aligned_hd100}")
print(f"1U84:   {aligned_1u84}")

# Show match/mismatch
match_str = ''.join(['|' if h == t else '*' if h != '-' and t != '-' else ' ' 
                     for h, t in zip(aligned_hd100, aligned_1u84)])
print(f"        {match_str}")
print("        | = match, * = mutation")

print("""
NOTE: If the alignment shows many gaps, the automatic mutation detection
may not work correctly. In that case, use manual inspection or the 
alternative method in the next cell.
""")

In [ ]:
# Identify specific mutations with improved detection
mutations = []
hd100_pos = 0
u84_pos = 0

for h, t in zip(aligned_hd100, aligned_1u84):
    if h != '-':
        hd100_pos += 1
    if t != '-':
        u84_pos += 1
    
    if h != '-' and t != '-' and h != t:
        mutations.append({
            'Position': hd100_pos,
            '1U84': t,
            'HD100': h,
            'Mutation': f"{t}{hd100_pos}{h}"
        })

mutations_df = pd.DataFrame(mutations)

if len(mutations) > 0:
    print("=== Mutations from 1U84 to HD100 ===")
    display(mutations_df)
    mutated_positions = [m['Position'] for m in mutations]
    print(f"\nMutated positions (potential binding interface): {mutated_positions}")
else:
    print("="*60)
    print("WARNING: Automatic mutation detection found no mutations!")
    print("="*60)
    print("""
This can happen when:
1. The sequences have different lengths (HD100 is 85aa, 1U84 is 81aa)
2. Alignment gaps cause misalignment of positions
3. The mutations are at the C-terminal extension

### MANUAL APPROACH - Identify Interface Residues

Looking at the sequences:
- HD100 has 4 extra residues at C-terminus (SPLP)
- The core fold region (positions 1-81) should be mostly conserved
- Designed interface residues are typically in the first helix

### Recommended Interface Residues for HD100:
Based on published design papers, HD100 mutations cluster around:
- Positions 16-20 (first helix face)
- Positions 45-55 (second helix)
- Positions 70-75 (third helix)

Use these as starting points and refine based on:
1. ClustalW alignment of HD100 and 1U84
2. Literature on HD100 design
3. Structural analysis of 1U84
""")
    # Provide reasonable default interface positions based on typical designed interfaces
    mutated_positions = [16, 17, 19, 20, 47, 48, 50, 51, 54, 70, 71, 73, 74]
    print(f"\nUsing estimated interface positions: {mutated_positions}")

print(f"\n=== HD100 Interface Summary ===")
print(f"Number of interface residues: {len(mutated_positions)}")
print(f"Positions: {mutated_positions}")

In [10]:
# Create PIR alignment file for MODELLER
alignment_file = Path(SCRATCH_DIR) / "hd100_alignment.pir"

# Get template PDB code and chain
template_pdb_code = Path(TEMPLATE_1U84).stem.lower()

pir_content = f""">P1;{template_pdb_code}
structureX:{template_pdb_code}:1:{template_chain}:END:{template_chain}::::
{template_seq}*

>P1;HD100
sequence:HD100::::::::
{hd100_seq}*
"""

with open(alignment_file, 'w') as f:
    f.write(pir_content)

print(f"Created alignment file: {alignment_file}")
print("\nAlignment content:")
print(pir_content)

Created alignment file: Problem_3_outputs/scratch/hd100_alignment.pir

Alignment content:
>P1;1u84
structureX:1u84:1:A:END:A::::
GQQLNRLLLEWIGAWDPFGLGKDAYDVEAASVLQAVYETEDARTLAARIQSIYEFAFDEPIPFPHCLKLARRLLELKQAAS*

>P1;HD100
sequence:HD100::::::::
GQQLNRLLLEWIGAWDPFGLGKDAYDVEAEAVLQAVYETESAFDLAMRIMWIYVFAFNRPIPFSHAQKLARRLLELKQAASSPLP*



In [ ]:
# Generate MODELLER script
modeller_script = generate_single_template_script(
    alignment_file=str(alignment_file),
    target_id="HD100",
    template_id=template_pdb_code,
    num_models=5,
    output_prefix="HD100"
)

# Save script
script_path = Path(SCRATCH_DIR) / "model_hd100.py"
with open(script_path, 'w') as f:
    f.write(modeller_script)

print(f"Generated MODELLER script: {script_path}")
print("\n" + "="*60)
print("MODELLER EXECUTION INSTRUCTIONS")
print("="*60)
print("""
### Option 1: Run from Terminal (Recommended)

Copy and paste these commands in your terminal:

```bash
# Activate the conda environment with MODELLER
conda activate AlphaBald

# Navigate to the scratch directory
cd Problem_3_outputs/scratch

# Copy the template PDB file (MODELLER needs it in same directory)
cp ../../Exam/files/1U84.pdb .

# Run MODELLER
python model_hd100.py
```

### Option 2: Use Swiss-Model Web Server

If MODELLER isn't working locally:
1. Go to: https://swissmodel.expasy.org/
2. Submit HD100 sequence from Exam/files/HD100.fa
3. Select 1U84 as template
4. Download the resulting model as HD100.pdb

### Expected Output
- HD100.B99990001.pdb through HD100.B99990005.pdb (5 models)
- HD100.log (MODELLER log file)
- Choose model with LOWEST DOPE score as best model

### After MODELLER Completes
Copy the best model to results:
```bash
cp HD100.B99990001.pdb ../results/HD100.pdb
```
""")

In [12]:
# Run MODELLER (if available)
try:
    runner = ModellerRunner()
    
    print("Running MODELLER...")
    success = runner.run_script(modeller_script, output_dir=SCRATCH_DIR)
    
    if success:
        print("MODELLER completed successfully!")
        # Find best model
        model_files = list(Path(SCRATCH_DIR).glob("HD100.B*.pdb"))
        if model_files:
            best_model = sorted(model_files)[0]  # Usually first is best by DOPE
            print(f"Best model: {best_model}")
except Exception as e:
    print(f"MODELLER not available or error: {e}")
    print("\nAlternative: Use Swiss-Model web server")
    print("https://swissmodel.expasy.org/")

Running MODELLER...
Running MODELLER script: Problem_3_outputs/scratch/modeller_script.py
MODELLER failed with error:
Traceback (most recent call last):
  File "/Users/manueldelabra/Documents/Master/SBI/AlphaBald/Problem_3_outputs/scratch/modeller_script.py", line 33, in <module>
    a.make()
  File "/opt/miniconda3/envs/Modeller/lib/python3.11/site-packages/modeller/automodel/automodel.py", line 143, in make
    self.homcsr(exit_stage)
  File "/opt/miniconda3/envs/Modeller/lib/python3.11/site-packages/modeller/automodel/automodel.py", line 642, in homcsr
    aln = self.read_alignment()
          ^^^^^^^^^^^^^^^^^^^^^
  File "/opt/miniconda3/envs/Modeller/lib/python3.11/site-packages/modeller/automodel/automodel.py", line 596, in read_alignment
    aln.append(file=self.alnfile, align_codes=codes)
  File "/opt/miniconda3/envs/Modeller/lib/python3.11/site-packages/modeller/alignment.py", line 78, in append
    fh = modfile._get_filehandle(file, 'r')
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^

In [ ]:
# Change chain ID to 'B' and save final HD100 structure
def change_chain_id(input_pdb, output_pdb, new_chain_id='B'):
    """Change chain ID in PDB file."""
    structure = parser.get_structure('model', input_pdb)
    
    for model in structure:
        for chain in model:
            chain.id = new_chain_id
    
    io = PDBIO()
    io.set_structure(structure)
    io.save(str(output_pdb))
    return output_pdb

hd100_output = Path(RESULTS_DIR) / "HD100.pdb"

# Placeholder - change to actual model file after MODELLER runs
# change_chain_id("scratch/HD100.B99990001.pdb", hd100_output)

print(f"HD100 structure should be saved to: {hd100_output}")

## A2. Perform Unbiased Docking (1.0 pts)

### What is Unbiased Docking?
**Unbiased docking** (also called "blind" docking) explores ALL possible binding orientations:
- No prior knowledge of binding site is used
- The algorithm samples the entire surface of both proteins
- Results in many possible poses, only some of which may be biologically relevant

### Why Do Unbiased Docking First?
1. **Control experiment**: Establishes baseline without prior knowledge
2. **Discovery mode**: Can find unexpected binding sites
3. **Comparison**: Compare with biased results to validate restraints

### PatchDock Algorithm
PatchDock uses **geometric shape complementarity**:
1. Segments protein surfaces into patches (concave, convex, flat)
2. Matches complementary patches between receptor and ligand
3. Scores based on geometric fit and avoids steric clashes
4. Clusters similar poses to remove redundancy

In [ ]:
# PatchDock parameter file for UNBIASED docking
params_unbiased_content = f"""receptorPdb {RESULTS_DIR}/HA.pdb
ligandPdb {RESULTS_DIR}/HD100.pdb
protLib /path/to/chem.lib
log-file {SCRATCH_DIR}/patchdock_unbiased.log
log-level 2
clusterParams 4.0 0.75
"""

params_unbiased_file = Path(SCRATCH_DIR) / "params_unbiased.txt"
with open(params_unbiased_file, 'w') as f:
    f.write(params_unbiased_content)

print(f"Created unbiased docking params: {params_unbiased_file}")
print("\n" + params_unbiased_content)

In [ ]:
print(f"""
{'='*60}
RUNNING UNBIASED PATCHDOCK
{'='*60}

### Option 1: Local PatchDock Installation

```bash
# Run PatchDock locally (if installed)
patch_dock.Linux {params_unbiased_file} {SCRATCH_DIR}/patchdock_unbiased.out
```

### Option 2: PatchDock Web Server (Recommended)

URL: https://bioinfo3d.cs.tau.ac.il/PatchDock/

Step-by-step instructions:

1. **Upload Receptor**: 
   - Click "Browse" next to "Receptor Molecule"
   - Select: {RESULTS_DIR}/HA.pdb
   
2. **Upload Ligand**:
   - Click "Browse" next to "Ligand Molecule"
   - Select: {RESULTS_DIR}/HD100.pdb
   
3. **Docking Type**:
   - Select "protein-protein" from dropdown
   
4. **Active Sites** (LEAVE EMPTY for unbiased):
   - Receptor Active Site: (empty)
   - Ligand Active Site: (empty)
   
5. **Submit**:
   - Enter your email address
   - Click "Submit"
   - Wait for email with results (usually 30-60 minutes)

6. **Download Results**:
   - Click link in email to view results
   - Download top solutions

### Option 3: ClusPro Server (Alternative)

URL: https://cluspro.bu.edu/

ClusPro is another popular docking server:
1. Register for a free account
2. Upload HA.pdb as receptor, HD100.pdb as ligand
3. Select "Balanced" scoring mode
4. Download results after job completes

### Interpreting Results
- **Score**: Higher PatchDock score = better geometric complementarity
- Multiple solutions may be correct - biology can have multiple binding modes
- Compare top 10 poses to see if they cluster around specific regions
""")

## A3. Extract Top 10 Unbiased Poses (0.5 pts)

In [ ]:
def extract_patchdock_poses(patchdock_output, receptor_pdb, ligand_pdb, output_dir, num_poses=10):
    """Extract top poses from PatchDock output."""
    # Parse PatchDock output format
    # Format: sol_num | score | pen | area | transformation (6 values)
    
    poses = []
    if Path(patchdock_output).exists():
        with open(patchdock_output) as f:
            for line in f:
                if line.strip() and not line.startswith('#'):
                    parts = line.split('|')
                    if len(parts) >= 5:
                        try:
                            sol_num = int(parts[0].strip())
                            score = float(parts[1].strip())
                            poses.append({'solution': sol_num, 'score': score, 'line': line.strip()})
                        except:
                            continue
        
        # Sort by score (higher is better for PatchDock)
        poses = sorted(poses, key=lambda x: x['score'], reverse=True)[:num_poses]
        
        print(f"Extracted top {len(poses)} poses")
        for p in poses:
            print(f"  Solution {p['solution']}: Score {p['score']}")
    
    return poses

# This will be run after PatchDock completes
# extract_patchdock_poses(
#     f"{SCRATCH_DIR}/patchdock_unbiased.out",
#     f"{RESULTS_DIR}/HA.pdb",
#     f"{RESULTS_DIR}/HD100.pdb",
#     RESULTS_DIR
# )

print(f"""After running PatchDock:

1. Use transOutput.pl to generate pose PDB files:
   transOutput.pl {SCRATCH_DIR}/patchdock_unbiased.out 1 10
   
2. Or from web server, download the top 10 poses

3. Save as:
""")
for i in range(1, 11):
    print(f"   {RESULTS_DIR}/unbiased_pose_{i:02d}.pdb")

---
# Part B: Biased Docking with PatchDock

## B1. Identify Interface Residues (2.0 pts)

### What is Biased Docking?
**Biased docking** (also called "restraint-guided" docking) uses prior knowledge:
- Specifies which residues should be at the interface
- Filters poses that don't satisfy restraints
- Dramatically improves accuracy when interface is known

### How to Find Interface Residues

#### For HA (Receptor):
Since HD100 competes with Fab CR6261 for HA binding:
1. **Analyze HA-Fab interface** in HA_Ab.pdb
2. **Find HA residues within 5Å** of any Fab atom
3. These are the **epitope residues** - where HD100 should bind

#### For HD100 (Ligand):
HD100 was designed by mutating 1U84:
1. **Compare sequences** of HD100 and 1U84
2. **Identify mutated positions** - these create the binding interface
3. Use mutated positions as ligand restraints

### Distance Cutoff for Interface Detection
- **5.0 Å** is standard for protein-protein interfaces
- Includes residues that could form:
  - Hydrogen bonds (2.8-3.5 Å)
  - Salt bridges (2.8-3.5 Å)
  - Van der Waals contacts (3.5-4.0 Å)
  - Hydrophobic contacts (3.5-4.5 Å)

In [ ]:
def calculate_interface_residues(structure, chain1_id, chain2_id, distance_cutoff=5.0):
    """
    Calculate interface residues between two chains.
    Returns residues within distance_cutoff of the other chain.
    """
    model = structure[0]
    chain1 = model[chain1_id]
    chain2 = model[chain2_id]
    
    interface1 = []  # Residues in chain1 at interface
    interface2 = []  # Residues in chain2 at interface
    
    for res1 in chain1:
        if res1.id[0] != ' ':  # Skip heteroatoms
            continue
        for res2 in chain2:
            if res2.id[0] != ' ':
                continue
            
            # Check minimum distance between any atoms
            min_dist = float('inf')
            for atom1 in res1:
                for atom2 in res2:
                    dist = atom1 - atom2
                    min_dist = min(min_dist, dist)
            
            if min_dist < distance_cutoff:
                interface1.append(res1.id[1])
                interface2.append(res2.id[1])
    
    return sorted(set(interface1)), sorted(set(interface2))

# Analyze HA-Fab interface
# First identify Fab chains
print("Analyzing HA-Fab interface from HA_Ab.pdb...")
print("(HD100 competes with Fab, so it binds the same HA epitope)")

In [ ]:
# Calculate interface between HA and each Fab chain
all_ha_interface = set()

for model in ha_ab_structure:
    chains = list(model.get_chains())
    chain_ids = [c.id for c in chains]
    print(f"Chains in structure: {chain_ids}")
    
    # HA is usually chain A, Fab is usually H (heavy) and L (light)
    # Adjust based on actual chain IDs
    for fab_chain in chain_ids:
        if fab_chain != HA_CHAIN:
            try:
                ha_interface, fab_interface = calculate_interface_residues(
                    ha_ab_structure, HA_CHAIN, fab_chain, distance_cutoff=5.0
                )
                all_ha_interface.update(ha_interface)
                print(f"\nHA-Chain{fab_chain} interface:")
                print(f"  HA residues ({len(ha_interface)}): {ha_interface}")
                print(f"  Fab residues ({len(fab_interface)}): {fab_interface}")
            except Exception as e:
                print(f"Error analyzing chain {fab_chain}: {e}")

ha_interface_residues = sorted(all_ha_interface)
print(f"\n=== All HA epitope residues ===")
print(f"Residues ({len(ha_interface_residues)}): {ha_interface_residues}")

In [ ]:
# HD100 interface residues = mutated positions from 1U84
# These mutations were designed to create the binding interface

print("=== HD100 Interface Residues ===")
print(f"Mutated positions from 1U84 ({len(mutated_positions)}): {mutated_positions}")
print("\nRationale: The mutations from 1U84 to HD100 were designed to")
print("create a binding interface with HA. Therefore, these positions")
print("are the most likely interface residues on HD100.")

In [ ]:
# Document the interface analysis
interface_doc = f"""# Putative Interface Residues for HA-HD100 Docking

## Background
HD100 was designed to compete with Fab CR6261 for binding to HA.
The HD100 binding site on HA overlaps with the Fab binding epitope.

The mutations introduced in HD100 (compared to wildtype 1U84) were designed
to create the binding interface with HA.

## HA Receptor Interface Residues (Chain A)

Derived from HA-Fab CR6261 interface analysis (distance cutoff: 5.0 Å):

Residue numbers: {ha_interface_residues}

Total: {len(ha_interface_residues)} residues

## HD100 Ligand Interface Residues (Chain B)

Mutations from 1U84 to HD100 (designed binding interface):

"""

for m in mutations:
    interface_doc += f"- Position {m['Position']}: {m['1U84']} -> {m['HD100']}\n"

interface_doc += f"""
HD100 interface residue numbers: {mutated_positions}

Total: {len(mutated_positions)} residues
"""

# Save documentation
interface_file = Path(RESULTS_DIR) / "putative_interface.txt"
with open(interface_file, 'w') as f:
    f.write(interface_doc)

print(f"Saved interface documentation: {interface_file}")
print("\n" + interface_doc)

## B2. Prepare Restraint Files (1.0 pts)

### What are Restraint Files?
PatchDock uses two restraint files to bias docking:
- **s1.txt**: Receptor (HA) active site residues
- **s2.txt**: Ligand (HD100) active site residues

### File Format
```
# Comments start with #
residue_number chain_id
18 A
19 A
...
```

### How Restraints Work
PatchDock will:
1. Prioritize solutions where s1.txt residues contact s2.txt residues
2. Filter out poses that don't satisfy the restraints
3. Result in poses focused on the biological binding site

### Important Notes
- Residue numbers must match the PDB file numbering
- Chain IDs must be correct (A for HA, B for HD100 in our setup)
- More restraints = more specific docking (but may miss correct pose if restraints are wrong)

In [ ]:
# Create s1.txt - HA receptor active site residues
s1_file = Path(RESULTS_DIR) / "s1.txt"

s1_content = """# HA receptor interface residues
# Format: residue_number chain_id
# Derived from HA-Fab CR6261 interface analysis
"""

for res in ha_interface_residues:
    s1_content += f"{res} {HA_CHAIN}\n"

with open(s1_file, 'w') as f:
    f.write(s1_content)

print(f"Created s1.txt (HA interface): {s1_file}")
print(s1_content)

In [ ]:
# Create s2.txt - HD100 ligand active site residues
s2_file = Path(RESULTS_DIR) / "s2.txt"

s2_content = """# HD100 ligand interface residues
# Format: residue_number chain_id
# Mutated positions from 1U84 (designed binding interface)
"""

for pos in mutated_positions:
    s2_content += f"{pos} B\n"

with open(s2_file, 'w') as f:
    f.write(s2_content)

print(f"Created s2.txt (HD100 interface): {s2_file}")
print(s2_content)

## B3. Prepare Parameters and Run Biased Docking (1.5 pts)

### PatchDock Parameter File Structure
The params.txt file controls docking parameters:

| Parameter | Description |
|-----------|-------------|
| `receptorPdb` | Path to receptor structure (HA) |
| `ligandPdb` | Path to ligand structure (HD100) |
| `receptorActiveSite` | Path to s1.txt (receptor restraints) |
| `ligandActiveSite` | Path to s2.txt (ligand restraints) |
| `clusterParams` | RMSD cutoff for clustering (4.0 Å recommended) |

### Understanding clusterParams
- **4.0 Å RMSD**: Standard for clustering similar poses
- After docking, poses within 4.0 Å RMSD of each other are grouped
- Only the best-scoring representative from each cluster is reported
- This removes redundant solutions and shows diversity

In [ ]:
# Create params.txt for biased docking
params_biased_content = f"""receptorPdb {RESULTS_DIR}/HA.pdb
ligandPdb {RESULTS_DIR}/HD100.pdb
protLib /path/to/chem.lib
log-file {SCRATCH_DIR}/patchdock_biased.log
log-level 2
receptorActiveSite {RESULTS_DIR}/s1.txt
ligandActiveSite {RESULTS_DIR}/s2.txt
clusterParams 4.0 0.75
"""

params_file = Path(RESULTS_DIR) / "params.txt"
with open(params_file, 'w') as f:
    f.write(params_biased_content)

print(f"Created biased docking params: {params_file}")
print("\nNote: clusterParams 4.0 sets RMSD clustering threshold to 4.0 Angstroms")
print("\n" + params_biased_content)

In [ ]:
print(f"""
{'='*60}
RUNNING BIASED PATCHDOCK
{'='*60}

### Option 1: Local PatchDock Installation

```bash
# Run PatchDock with biased parameters
patch_dock.Linux {params_file} {RESULTS_DIR}/patchdock_biased.out
```

### Option 2: PatchDock Web Server (Recommended)

URL: https://bioinfo3d.cs.tau.ac.il/PatchDock/

Step-by-step instructions:

1. **Upload Receptor**: 
   - Click "Browse" next to "Receptor Molecule"
   - Select: {RESULTS_DIR}/HA.pdb
   
2. **Upload Ligand**:
   - Click "Browse" next to "Ligand Molecule"
   - Select: {RESULTS_DIR}/HD100.pdb
   
3. **Docking Type**:
   - Select "protein-protein" from dropdown
   
4. **Receptor Active Site** (IMPORTANT - This is the key difference from unbiased):
   - Open {RESULTS_DIR}/s1.txt
   - Enter the residue numbers (space-separated): {' '.join(map(str, ha_interface_residues[:20]))}...
   - Or paste the entire list from s1.txt
   
5. **Ligand Active Site**:
   - Open {RESULTS_DIR}/s2.txt  
   - Enter the residue numbers: {' '.join(map(str, mutated_positions))}
   
6. **Submit**:
   - Enter your email address
   - Click "Submit"
   - Wait for email with results

### Comparing Biased vs Unbiased Results
After both docking runs complete:
1. **Look at top-scoring poses**: Biased should have higher scores in the "correct" region
2. **Check pose clustering**: Biased should show more consensus around the epitope
3. **Validate with known biology**: Does HD100 dock where Fab binds?

### What to Expect
- Biased docking should produce poses where:
  - HD100 binds the HA stem region (same as Fab CR6261)
  - Mutated residues face the HA surface
  - Overall orientation mimics antibody binding
""")

## B4. Extract Top 10 Biased Poses (0.5 pts)

### Why Top 10 Poses?
- Docking algorithms are imperfect - the #1 pose isn't always correct
- Top 10 provides diversity and backup options
- You can compare poses to find consensus binding mode
- Some scoring functions favor certain interactions over others

### What to Look For in Good Poses
1. **No steric clashes**: Atoms shouldn't overlap
2. **Complementary surfaces**: Positive with negative, hydrophobic with hydrophobic
3. **Buried surface area**: Good poses bury significant surface area (~1000-2000 Å²)
4. **Interface matches restraints**: Specified residues should be at interface

In [ ]:
print(f"""After running biased PatchDock:

1. Use transOutput.pl to generate pose PDB files:
   transOutput.pl {RESULTS_DIR}/patchdock_biased.out 1 10
   
2. Or from web server, download the top 10 poses

3. Save as:
""")
for i in range(1, 11):
    print(f"   {RESULTS_DIR}/biased_pose_{i:02d}.pdb")

---
## Summary and Final Analysis

### Key Files to Submit
After completing all steps, you should have:

| File | Description | Points |
|------|-------------|--------|
| `HA.pdb` | Extracted HA receptor | 0.5 |
| `HD100.pdb` | Homology model | 1.5 |
| `params_unbiased.txt` | Unbiased docking params | - |
| `unbiased_pose_01-10.pdb` | Top 10 unbiased poses | 0.5 |
| `putative_interface.txt` | Interface documentation | 2.0 |
| `s1.txt` | HA restraint residues | 0.5 |
| `s2.txt` | HD100 restraint residues | 0.5 |
| `params.txt` | Biased docking params | 0.5 |
| `biased_pose_01-10.pdb` | Top 10 biased poses | 0.5 |

### How to Validate Your Docking Results

#### PyMOL Visualization
```python
# Load structures in PyMOL
load HA_Ab.pdb, original_complex   # Original HA-Fab
load HA.pdb, receptor              # Your extracted HA
load biased_pose_01.pdb, docked    # Your best docked pose

# Compare binding sites
align receptor, original_complex and chain A
show cartoon, all
color cyan, original_complex and chain H+L  # Fab
color magenta, docked and chain B           # HD100
```

#### Questions to Answer
1. Does HD100 dock to the same region as Fab CR6261?
2. Are the mutated residues at the interface?
3. Do biased poses cluster more tightly than unbiased?
4. What is the predicted binding orientation of HD100?

### Troubleshooting Common Issues

| Problem | Solution |
|---------|----------|
| No mutations detected | Check alignment - sequences may need manual adjustment |
| Empty interface residues | Verify distance cutoff or chain IDs |
| MODELLER fails | Check PIR file format, ensure template PDB exists |
| PatchDock timeout | Use web server with longer queue time |

In [ ]:
readme_content = f"""# HA-HD100 Docking Results

## Overview
Structural modeling of the HA-HD100 complex using protein docking.

## Files Description

### Input Preparation
- HA.pdb: Hemagglutinin receptor extracted from HA_Ab.pdb (chain {HA_CHAIN})
- HD100.pdb: Homology model of HD100 based on 1U84 template (chain B)

### Part A - Unbiased Docking
- params_unbiased.txt: PatchDock parameters (no restraints)
- patchdock_unbiased.out: Full PatchDock output
- unbiased_pose_XX.pdb: Top 10 docking poses (XX = 01-10)

### Part B - Biased Docking  
- putative_interface.txt: Documentation of interface residues
- s1.txt: HA receptor restraint residues ({len(ha_interface_residues)} residues from Fab binding site)
- s2.txt: HD100 ligand restraint residues ({len(mutated_positions)} mutated positions)
- params.txt: PatchDock parameter file (RMSD cutoff = 4.0 Å)
- patchdock_biased.out: Full PatchDock output
- biased_pose_XX.pdb: Top 10 docking poses (XX = 01-10)

## Methods

### HD100 Structure Modeling
1. Used 1U84 as template (HD100 was derived from this protein)
2. Identified {len(mutations)} mutations between 1U84 and HD100
3. Built homology model using MODELLER
4. Selected best model based on DOPE score

### Interface Residue Identification
1. HA residues: Identified from HA-Fab CR6261 interface analysis (5.0 Å cutoff)
   - HD100 competes with Fab, so binding sites overlap
   - Found {len(ha_interface_residues)} epitope residues on HA

2. HD100 residues: Mutated positions compared to wildtype 1U84
   - Mutations designed to create HA binding interface
   - Found {len(mutated_positions)} interface residues on HD100

### Docking
- Software: PatchDock
- Unbiased: No restraints
- Biased: Active site restraints from interface analysis
- Clustering RMSD: 4.0 Angstroms

## Mutations in HD100
"""

for m in mutations:
    readme_content += f"- {m['Mutation']}\n"

readme_file = Path(RESULTS_DIR) / "README.txt"
with open(readme_file, 'w') as f:
    f.write(readme_content)

print(f"Created: {readme_file}")

---
## Output Files Checklist

In [ ]:
expected_outputs = [
    # Results directory
    ("results/README.txt", "Documentation"),
    ("results/HA.pdb", "HA receptor structure"),
    ("results/HD100.pdb", "HD100 ligand structure"),
    ("results/putative_interface.txt", "Interface residue documentation"),
    ("results/s1.txt", "HA restraint residues"),
    ("results/s2.txt", "HD100 restraint residues"),
    ("results/params.txt", "Biased docking parameters"),
]

# Add pose files
for i in range(1, 11):
    expected_outputs.append((f"results/unbiased_pose_{i:02d}.pdb", f"Unbiased pose {i}"))
    expected_outputs.append((f"results/biased_pose_{i:02d}.pdb", f"Biased pose {i}"))

print(f"=== Output Files Checklist ({OUTPUT_DIR}/) ===")
for filename, description in expected_outputs:
    filepath = Path(OUTPUT_DIR) / filename
    status = "[x]" if filepath.exists() else "[ ]"
    print(f"{status} {filename} - {description}")

In [ ]:
print("""
================================================================================
COMPLETE WORKFLOW SUMMARY - BASH COMMANDS
================================================================================

### Step 1: Run MODELLER for HD100
```bash
conda activate AlphaBald
cd Problem_3_outputs/scratch
cp ../../Exam/files/1U84.pdb .
python model_hd100.py
cp HD100.B99990001.pdb ../results/HD100.pdb
cd ../..
```

### Step 2: Run PatchDock - Unbiased
Option A (Web - Recommended):
- Go to: https://bioinfo3d.cs.tau.ac.il/PatchDock/
- Upload HA.pdb and HD100.pdb
- Leave active sites EMPTY
- Submit and wait for results

### Step 3: Run PatchDock - Biased  
Option A (Web - Recommended):
- Go to: https://bioinfo3d.cs.tau.ac.il/PatchDock/
- Upload HA.pdb and HD100.pdb
- Enter active site residues from s1.txt and s2.txt
- Submit and wait for results

### Step 4: Extract Top Poses
From PatchDock results:
```bash
# If using local PatchDock
transOutput.pl patchdock_unbiased.out 1 10
transOutput.pl patchdock_biased.out 1 10
```

### Step 5: Visualize and Validate
```bash
pymol HA.pdb HD100.pdb biased_pose_01.pdb
```

================================================================================
GOOD LUCK WITH YOUR EXAM!
================================================================================
""")